In [ ]:
%pip install -q timm pydicom scikit-learn albumentations tqdm

In [ ]:
import sys, json, os
from pathlib import Path
import torch
import numpy as np
import pandas as pd

# Add source path (parent of src, not src itself)
sys.path.append('/kaggle/working/rsna_knee')

from src.data.dataloader import create_dataloaders, create_test_dataloader, LABELS
from src.models.model import create_model
from src.training.trainer import train_fold
from src.training.inference import run_inference_pipeline

DATA_ROOT = Path('/kaggle/input/rsna-knee-abnormality-detection')
OUTPUT_DIR = Path('/kaggle/working/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open('/kaggle/working/rsna_knee/configs/train_config.json') as f:
    config = json.load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'Config: {json.dumps(config, indent=2)}')

In [ ]:
# Cross-validation training on REAL labeled data only
fold_aucs = []

for fold in range(config['n_folds']):
    print(f'\n=== FOLD {fold + 1}/{config["n_folds"]} ===')
    
    train_loader, val_loader = create_dataloaders(
        DATA_ROOT,
        batch_size=config['batch_size'],
        num_workers=config['num_workers'],
        max_series=config['max_series'],
        max_slices=config['max_slices'],
        use_pseudo_labels=False,  # Using only true labels
        slice_sampling=config['slice_sampling'],
        window_method=config['window_method'],
        three_channel=config['three_channel'],
        channel_method=config['channel_method'],
    )
    
    auc = train_fold(fold, train_loader, val_loader, config, device, OUTPUT_DIR)
    fold_aucs.append(auc)
    
    del train_loader, val_loader
    torch.cuda.empty_cache()

print(f'\nFold AUCs: {fold_aucs}')
print(f'Mean CV AUC: {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}')

In [ ]:
# Ensemble inference on test set
fold_weights = {i: 1.0 for i in range(config['n_folds'])}

submission_path = run_inference_pipeline(
    DATA_ROOT,
    OUTPUT_DIR,
    OUTPUT_DIR / 'submission',
    config,
    fold_weights=fold_weights,
    use_tta=True,
    batch_size=config['batch_size'],
    num_workers=config['num_workers']
)

print(f'Submission ready: {submission_path}')

In [ ]:
# Verify submission format
sub_df = pd.read_csv(submission_path)
sample_df = pd.read_csv(DATA_ROOT / 'sample_submission.csv')

print(f'Submission shape: {sub_df.shape}')
print(f'Sample shape: {sample_df.shape}')
print(f'Columns match: {list(sub_df.columns) == list(sample_df.columns)}')
print(f'UIDs match: {sub_df["StudyInstanceUID"].tolist() == sample_df["StudyInstanceUID"].tolist()}')

print('\nPrediction statistics:')
for label in LABELS:
    print(f'  {label}: mean={sub_df[label].mean():.4f}, std={sub_df[label].std():.4f}, min={sub_df[label].min():.4f}, max={sub_df[label].max():.4f}')